In [1]:
import langchain
import faiss
import pypdf
import sentence_transformers
import openai

print("All libraries imported successfully!")

C:\Users\gondk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries imported successfully!


In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader

folder_path = "data/pdfs"

documents = []

for file in os.listdir(folder_path):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(folder_path, file))
        docs = loader.load()
        
        # add metadata
        for d in docs:
            d.metadata["source_file"] = file
            d.metadata["scheme"] = file.replace(".pdf","")

        documents.extend(docs)

print("Total pages loaded:", len(documents))

Multiple definitions in dictionary at byte 0x10dcaa7 for key /Info
Multiple definitions in dictionary at byte 0x10dcab4 for key /Info


Total pages loaded: 307


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total chunks created:", len(chunks))

Total chunks created: 1046


In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2893.18it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
import re

def clean_text(text):
    text = text.replace("\x00", "")
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    return text

for doc in chunks:
    doc.page_content = clean_text(doc.page_content)

In [7]:
!pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\gondk\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
from sqlalchemy import create_engine
from langchain_postgres import PGVector
from dotenv import load_dotenv
import os

# Load .env file
load_dotenv()

# Read variables from .env
host = os.getenv("SUPABASE_HOST")
port = os.getenv("SUPABASE_PORT")
database = os.getenv("SUPABASE_DB")
user = os.getenv("SUPABASE_USER")
password = os.getenv("SUPABASE_PASSWORD")

# Create database engine
engine = create_engine(
    "postgresql+psycopg2://",
    connect_args={
        "host": host,
        "port": port,
        "database": database,
        "user": user,
        "password": password
    }
)

vectorstore = PGVector.from_documents(
    documents=chunks,
    embedding=embeddings,
    connection=engine,
    collection_name="scholarship_docs"
)

print("✅ Vectors stored successfully in Supabase!")

✅ Vectors stored successfully in Supabase!


In [5]:
from groq import Groq
from dotenv import load_dotenv
import os

# ✅ Load env file
load_dotenv()

# ✅ Get key from env
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# ✅ Now print
print("GROQ KEY:", repr(GROQ_API_KEY))

GROQ KEY: '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'
